# Churn Analysis

### This analysis seeks to answer the following hypotheses: 
### 1. Do customers with late deliveries churn? 
### 2. Do low review scores increase churn? 
### 3. Do states with worse delivery times have lower retention? 
### 4. Do premium customers tolerate more delays? 
### 5. Does frequency affect satisfaction?

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd

In [8]:
load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

try:
    folder = r"C:\Professional_project\Churn_and_Marketing_Analytics\Script_SQL"
    file = "dataset_exploration.sql"
    path = os.path.join(folder, file)
    
    with open(path, "r", encoding="utf-8") as f:
        sql = f.read()

    engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')

    churn_df = pd.read_sql(sql, con=engine)
    print("The data has been successfully loaded!!!")
    
except FileNotFoundError:
    print(f"The file was not found in the path:{path}")

except Exception as e:
    print(f'--Error details--: {e}')

finally:
    engine.dispose()

The data has been successfully loaded!!!


### Dataset cleaning

In [10]:
churn_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 98666 entries, 0 to 98665
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               98666 non-null  str    
 1   avg_order_value           98666 non-null  float64
 2   customer_state            98666 non-null  str    
 3   total_orders              98666 non-null  int64  
 4   total_spent               98665 non-null  float64
 5   avg_payment_installments  98665 non-null  float64
 6   avg_review_score          97917 non-null  float64
 7   avg_delivery_delay_days   96476 non-null  float64
 8   last_purchase             98666 non-null  object 
 9   first_purchase            98666 non-null  object 
 10  customer_lifetime_days    98666 non-null  int64  
 11  recency_days              98666 non-null  int64  
 12  churn_flag                98666 non-null  int64  
dtypes: float64(5), int64(4), object(2), str(2)
memory usage: 9.8+ MB


In [12]:
churn_df.isnull().sum()

customer_id                    0
avg_order_value                0
customer_state                 0
total_orders                   0
total_spent                    1
avg_payment_installments       1
avg_review_score             749
avg_delivery_delay_days     2190
last_purchase                  0
first_purchase                 0
customer_lifetime_days         0
recency_days                   0
churn_flag                     0
dtype: int64

In [14]:
churn_df["last_purchase"] = pd.to_datetime(churn_df["last_purchase"])
churn_df["first_purchase"] = pd.to_datetime(churn_df["first_purchase"])

In [15]:
churn_df.describe()

,avg_order_value,total_orders,total_spent,avg_payment_installments,avg_review_score,avg_delivery_delay_days,last_purchase,first_purchase,customer_lifetime_days,recency_days,churn_flag
count,98666.000000,98666.0,98665.000000,98665.000000,97917.000000,96476.000000,98666,98666,98666.0,98666.000000,98666.000000
mean,214.662671,1.0,159.093160,2.931870,4.105055,-11.876881,2017-12-31 06:48:52,2017-12-31 06:48:52,0.0,245.716062,0.611406
min,10.070000,1.0,4.070000,0.000000,1.000000,-147.000000,2016-09-04 00:00:00,2016-09-04 00:00:00,0.0,0.000000,0.000000
25%,63.600000,1.0,61.050000,1.000000,4.000000,-17.000000,2017-09-12 00:00:00,2017-09-12 00:00:00,0.0,122.000000,0.000000
50%,112.600000,1.0,104.120000,2.000000,5.000000,-12.000000,2018-01-19 00:00:00,2018-01-19 00:00:00,0.0,227.000000,1.000000
75%,201.840000,1.0,175.770000,4.000000,5.000000,-7.000000,2018-05-04 00:00:00,2018-05-04 00:00:00,0.0,356.000000,1.000000
max,109312.640000,1.0,13664.080000,24.000000,5.000000,188.000000,2018-09-03 00:00:00,2018-09-03 00:00:00,0.0,729.000000,1.000000
std,645.717837,0.0,218.898589,2.714891,1.330316,10.183854,NaN,NaN,0.0,153.392938,0.487433


In [20]:
churn_df = churn_df.dropna(subset=["avg_delivery_delay_days"])

In [21]:
churn_df["avg_review_score"] = (
    churn_df["avg_review_score"]
    .fillna(churn_df["avg_review_score"].median())
)

In [22]:
churn_df.isnull().sum()

customer_id                 0
avg_order_value             0
customer_state              0
total_orders                0
total_spent                 1
avg_payment_installments    1
avg_review_score            0
avg_delivery_delay_days     0
last_purchase               0
first_purchase              0
customer_lifetime_days      0
recency_days                0
churn_flag                  0
dtype: int64